In [1]:
import numpy as np

class RNN:
    """
    A simple character-level Recurrent Neural Network implemented from scratch.
    """
    
    def __init__(self, input_size, hidden_size, output_size, learning_rate=0.01):
        """
        Initialize the RNN with random weights.
        
        Args:
            input_size: Size of input vocabulary (number of unique characters)
            hidden_size: Number of hidden units
            output_size: Size of output vocabulary (same as input_size for char-level)
            learning_rate: Learning rate for gradient descent
        """
        self.hidden_size = hidden_size
        self.learning_rate = learning_rate
        
        # Xavier initialization for weights
        self.Wxh = np.random.randn(hidden_size, input_size) * np.sqrt(2.0 / input_size)
        self.Whh = np.random.randn(hidden_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        self.Why = np.random.randn(output_size, hidden_size) * np.sqrt(2.0 / hidden_size)
        
        # Biases initialized to zero
        self.bh = np.zeros((hidden_size, 1))
        self.by = np.zeros((output_size, 1))
        
        # Memory variables for Adagrad optimizer
        self.mWxh = np.zeros_like(self.Wxh)
        self.mWhh = np.zeros_like(self.Whh)
        self.mWhy = np.zeros_like(self.Why)
        self.mbh = np.zeros_like(self.bh)
        self.mby = np.zeros_like(self.by)
    
    def forward(self, inputs, h_prev):
        """
        Forward pass through the RNN.
        
        Args:
            inputs: List of input indices (one-hot encoded internally)
            h_prev: Previous hidden state
            
        Returns:
            outputs: Output probabilities at each time step
            hidden_states: Hidden states at each time step
        """
        xs, hs, ys, ps = {}, {}, {}, {}
        hs[-1] = np.copy(h_prev)
        
        for t, idx in enumerate(inputs):
            # One-hot encode input
            xs[t] = np.zeros((self.Wxh.shape[1], 1))
            xs[t][idx] = 1
            
            # Hidden state: h_t = tanh(Wxh @ x_t + Whh @ h_{t-1} + bh)
            hs[t] = np.tanh(self.Wxh @ xs[t] + self.Whh @ hs[t-1] + self.bh)
            
            # Output: y_t = Why @ h_t + by
            ys[t] = self.Why @ hs[t] + self.by
            
            # Softmax probabilities
            ps[t] = np.exp(ys[t]) / np.sum(np.exp(ys[t]))
        
        return xs, hs, ys, ps
    
    def backward(self, xs, hs, ps, targets):
        """
        Backward pass through time (BPTT).
        
        Args:
            xs: Input vectors at each time step
            hs: Hidden states at each time step
            ps: Output probabilities at each time step
            targets: Target indices at each time step
            
        Returns:
            Gradients for all parameters and loss
        """
        dWxh = np.zeros_like(self.Wxh)
        dWhh = np.zeros_like(self.Whh)
        dWhy = np.zeros_like(self.Why)
        dbh = np.zeros_like(self.bh)
        dby = np.zeros_like(self.by)
        dh_next = np.zeros_like(hs[0])
        
        loss = 0
        
        for t in reversed(range(len(targets))):
            # Cross-entropy loss
            loss += -np.log(ps[t][targets[t], 0] + 1e-8)
            
            # Gradient of loss w.r.t. output
            dy = np.copy(ps[t])
            dy[targets[t]] -= 1  # Softmax + cross-entropy gradient
            
            # Gradients for output layer
            dWhy += dy @ hs[t].T
            dby += dy
            
            # Gradient w.r.t. hidden state
            dh = self.Why.T @ dy + dh_next
            
            # Gradient through tanh
            dh_raw = (1 - hs[t] ** 2) * dh
            
            # Gradients for hidden layer
            dbh += dh_raw
            dWxh += dh_raw @ xs[t].T
            dWhh += dh_raw @ hs[t-1].T
            
            # Gradient for next iteration
            dh_next = self.Whh.T @ dh_raw
        
        # Clip gradients to prevent exploding gradients
        for dparam in [dWxh, dWhh, dWhy, dbh, dby]:
            np.clip(dparam, -5, 5, out=dparam)
        
        return dWxh, dWhh, dWhy, dbh, dby, loss
    
    def update_parameters(self, dWxh, dWhh, dWhy, dbh, dby):
        """
        Update parameters using Adagrad optimizer.
        """
        for param, dparam, mem in zip(
            [self.Wxh, self.Whh, self.Why, self.bh, self.by],
            [dWxh, dWhh, dWhy, dbh, dby],
            [self.mWxh, self.mWhh, self.mWhy, self.mbh, self.mby]
        ):
            mem += dparam ** 2
            param -= self.learning_rate * dparam / (np.sqrt(mem) + 1e-8)
    
    def sample(self, seed_idx, h, length=100):
        """
        Generate a sequence of characters starting from a seed.
        
        Args:
            seed_idx: Index of the seed character
            h: Initial hidden state
            length: Number of characters to generate
            
        Returns:
            List of generated character indices
        """
        x = np.zeros((self.Wxh.shape[1], 1))
        x[seed_idx] = 1
        indices = []
        
        for _ in range(length):
            h = np.tanh(self.Wxh @ x + self.Whh @ h + self.bh)
            y = self.Why @ h + self.by
            p = np.exp(y) / np.sum(np.exp(y))
            
            # Sample from the distribution
            idx = np.random.choice(range(p.shape[0]), p=p.ravel())
            indices.append(idx)
            
            # Prepare next input
            x = np.zeros((self.Wxh.shape[1], 1))
            x[idx] = 1
        
        return indices


class TextPreprocessor:
    """
    Handles text preprocessing for character-level language modeling.
    """
    
    def __init__(self, text):
        self.text = text
        self.chars = sorted(list(set(text)))
        self.vocab_size = len(self.chars)
        self.char_to_idx = {ch: i for i, ch in enumerate(self.chars)}
        self.idx_to_char = {i: ch for i, ch in enumerate(self.chars)}
    
    def encode(self, text):
        """Convert text to list of indices."""
        return [self.char_to_idx[ch] for ch in text]
    
    def decode(self, indices):
        """Convert list of indices back to text."""
        return ''.join([self.idx_to_char[i] for i in indices])


def train_rnn(corpus, hidden_size=100, seq_length=25, epochs=1000, learning_rate=0.1):
    """
    Train the RNN on a given corpus.
    
    Args:
        corpus: Text corpus for training
        hidden_size: Number of hidden units
        seq_length: Length of each training sequence
        epochs: Number of training iterations
        learning_rate: Learning rate
        
    Returns:
        Trained RNN model and preprocessor
    """
    # Preprocess text
    preprocessor = TextPreprocessor(corpus)
    vocab_size = preprocessor.vocab_size
    
    print(f"Corpus length: {len(corpus)} characters")
    print(f"Vocabulary size: {vocab_size} unique characters")
    print(f"Vocabulary: {preprocessor.chars}")
    print("-" * 50)
    
    # Initialize RNN
    rnn = RNN(vocab_size, hidden_size, vocab_size, learning_rate)
    
    # Training loop
    pointer = 0
    smooth_loss = -np.log(1.0 / vocab_size) * seq_length  # Initial loss estimate
    
    for epoch in range(epochs):
        # Reset if we've gone through the entire corpus
        if pointer + seq_length + 1 >= len(corpus) or epoch == 0:
            h_prev = np.zeros((hidden_size, 1))
            pointer = 0
        
        # Get input and target sequences
        inputs = preprocessor.encode(corpus[pointer:pointer + seq_length])
        targets = preprocessor.encode(corpus[pointer + 1:pointer + seq_length + 1])
        
        # Forward pass
        xs, hs, ys, ps = rnn.forward(inputs, h_prev)
        
        # Backward pass
        dWxh, dWhh, dWhy, dbh, dby, loss = rnn.backward(xs, hs, ps, targets)
        
        # Update parameters
        rnn.update_parameters(dWxh, dWhh, dWhy, dbh, dby)
        
        # Update smooth loss for monitoring
        smooth_loss = smooth_loss * 0.999 + loss * 0.001
        
        # Save last hidden state for next iteration
        h_prev = hs[len(inputs) - 1]
        
        # Print progress and sample
        if epoch % 100 == 0:
            print(f"Epoch {epoch}, Loss: {smooth_loss:.4f}")
            
            # Generate sample text
            sample_h = np.zeros((hidden_size, 1))
            sample_indices = rnn.sample(inputs[0], sample_h, length=50)
            sample_text = preprocessor.decode(sample_indices)
            print(f"Sample: {sample_text}")
            print("-" * 50)
        
        pointer += seq_length
    
    return rnn, preprocessor


# Main execution
if __name__ == "__main__":
    # Small corpus for demonstration
    corpus = """
    The quick brown fox jumps over the lazy dog.
    A journey of a thousand miles begins with a single step.
    To be or not to be, that is the question.
    All that glitters is not gold.
    The only thing we have to fear is fear itself.
    In the beginning was the Word, and the Word was with God.
    It was the best of times, it was the worst of times.
    To infinity and beyond!
    May the force be with you.
    Elementary, my dear Watson.
    I think, therefore I am.
    The truth is out there.
    Life is like a box of chocolates.
    Houston, we have a problem.
    Here's looking at you, kid.
    You can't handle the truth!
    There's no place like home.
    I'll be back.
    """
    
    print("=" * 50)
    print("RNN Training on Small Corpus")
    print("=" * 50)
    
    # Train the model
    rnn, preprocessor = train_rnn(
        corpus,
        hidden_size=128,
        seq_length=25,
        epochs=2000,
        learning_rate=0.1
    )
    
    # Generate final samples
    print("\n" + "=" * 50)
    print("Final Generated Samples")
    print("=" * 50)
    
    for i in range(3):
        h = np.zeros((128, 1))
        seed_char = corpus[np.random.randint(len(corpus))]
        seed_idx = preprocessor.char_to_idx.get(seed_char, 0)
        
        generated_indices = rnn.sample(seed_idx, h, length=100)
        generated_text = seed_char + preprocessor.decode(generated_indices)
        
        print(f"\nSample {i + 1} (seed: '{seed_char}'):")
        print(generated_text)

RNN Training on Small Corpus
Corpus length: 698 characters
Vocabulary size: 42 unique characters
Vocabulary: ['\n', ' ', '!', "'", ',', '.', 'A', 'E', 'G', 'H', 'I', 'L', 'M', 'T', 'W', 'Y', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']
--------------------------------------------------
Epoch 0, Loss: 93.4491
Sample: eab  eIiifkTfbqyuiiTxbxwvuIukTbYIq
uksbtyrIicfkTe.
--------------------------------------------------
Epoch 100, Loss: 94.4721
Sample:   m.me wuIthp.wttkT Ys c
cosso iIha  orLureaha Iu 
--------------------------------------------------
Epoch 200, Loss: 93.1165
Sample:   e,let  sG .aWi, k,WInc dh,   nz uoa  t.hsahm.dg!
--------------------------------------------------
Epoch 300, Loss: 91.2078
Sample:  titi,'    nectta nhd
glfT w th o fe t   th' eoksn
--------------------------------------------------
Epoch 400, Loss: 88.8352
Sample:   he be  uco,aaeehp ktHe uokg  .
iiu A esrho au.
 
------